In [1]:
from numpy import *

In [2]:
from torchvision import transforms, datasets
import torch
import time
from torch import nn
from torch.utils.data import DataLoader

In [4]:
# train_path = r"/scratch/mnist/train"
# test_path = r"/scratch/mnist/test"
train_path = r"C:\Users\hbse\projects\data\metrics\train"
test_path = r"C:\Users\hbse\projects\data\metrics\test"

mnist_transforms = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean=0.1307, std=0.3081)])
trainval_dataset = datasets.MNIST(root=train_path, train=True, download=False, transform=mnist_transforms)
test_dataset = datasets.MNIST(root=test_path, train=False, download=False, transform=mnist_transforms)

100.0%
100.0%
100.0%
100.0%
100.0%
100.0%
100.0%
100.0%


In [5]:
BATCH_SIZE = 32

trainval_dataloader = DataLoader(dataset=trainval_dataset, batch_size=BATCH_SIZE, shuffle=True) #, pin_memory=True)
test_dataloader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=True)
trainval_dataloader, test_dataloader

(<torch.utils.data.dataloader.DataLoader at 0x7f7d2922dfd0>,
 <torch.utils.data.dataloader.DataLoader at 0x7f7d29687f70>)

In [6]:
class LeNet5V1(nn.Module):
    def __init__(self):
        super().__init__()
        self.feature = nn.Sequential(
            #1
            nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5, stride=1, padding=2),   # 28*28->32*32-->28*28
            nn.Tanh(),
            nn.AvgPool2d(kernel_size=2, stride=2),  # 14*14
            
            #2
            nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5, stride=1),  # 10*10
            nn.Tanh(),
            nn.AvgPool2d(kernel_size=2, stride=2),  # 5*5
            
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=16*5*5, out_features=120),
            nn.Tanh(),
            nn.Linear(in_features=120, out_features=84),
            nn.Tanh(),
            nn.Linear(in_features=84, out_features=10),
        )
        
    def forward(self, x):
        return self.classifier(self.feature(x))
    

In [7]:
model_lenet5v1 = LeNet5V1()
model_lenet5v1

LeNet5V1(
  (feature): Sequential(
    (0): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): Tanh()
    (2): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (3): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
    (4): Tanh()
    (5): AvgPool2d(kernel_size=2, stride=2, padding=0)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=400, out_features=120, bias=True)
    (2): Tanh()
    (3): Linear(in_features=120, out_features=84, bias=True)
    (4): Tanh()
    (5): Linear(in_features=84, out_features=10, bias=True)
  )
)

In [8]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model_lenet5v1.parameters(), lr=0.001)

from torchmetrics import Accuracy
accuracy = Accuracy(task='multiclass', num_classes=10)  # e.g. accuracy(torch.Tensor([1,0,2]),torch.Tensor([1,1,2]))  # yields 2/3

### Load one batch per for loop

In [9]:
times = dict(load=[], device=[], eval_acc=[], bp=[])
times

{'load': [], 'device': [], 'eval_acc': [], 'bp': []}

In [10]:
# device-agnostic setup
device = 'cuda' if torch.cuda.is_available() else 'cpu'
accuracy = accuracy.to(device)
model_lenet5v1 = model_lenet5v1.to(device)

EPOCHS = 2

# for epoch in tqdm(range(EPOCHS)):
for epoch in range(EPOCHS):
    print(f"epoch - {epoch}")
    # Training loop
    train_loss, train_acc = 0.0, 0.0
    start = time.time()
    for X, y in trainval_dataloader:
        t_load = time.time()
        X, y = X.to(device), y.to(device)
        t_device = time.time()
        
        model_lenet5v1.train()
        
        y_pred = model_lenet5v1(X)
        
        loss = loss_fn(y_pred, y)
        train_loss += loss.item()
        
        acc = accuracy(y_pred, y)
        train_acc += acc
        t_eval_acc = time.time()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        t_bp = time.time()

        times["load"].append(t_load-start)
        times["device"].append(t_device-t_load)
        times["eval_acc"].append(t_eval_acc-t_device)
        times["bp"].append(t_bp-t_eval_acc)
        # print(f"t_load={t_load-start}\nt_device={t_device-t_load}\nt_eval_acc={t_eval_acc-t_device}\nt_bp={t_bp-t_eval_acc}")
        start = time.time()
        
    train_loss /= len(trainval_dataloader)
    train_acc /= len(trainval_dataloader)
        
    # Validation loop
    val_loss, val_acc = 0.0, 0.0
    model_lenet5v1.eval()
    with torch.inference_mode():
        for X, y in test_dataloader:
            X, y = X.to(device), y.to(device)
            
            y_pred = model_lenet5v1(X)
            
            loss = loss_fn(y_pred, y)
            val_loss += loss.item()
            
            acc = accuracy(y_pred, y)
            val_acc += acc
            
        val_loss /= len(test_dataloader)
        val_acc /= len(test_dataloader)
        
    print(f"Epoch: {epoch}| Train loss: {train_loss: .5f}| Train acc: {train_acc: .5f}| Test loss: {val_loss: .5f}| Test acc: {val_acc: .5f}")
print("done")

epoch - 0
Epoch: 0| Train loss:  0.19688| Train acc:  0.94255| Test loss:  0.07202| Test acc:  0.97694
epoch - 1
Epoch: 1| Train loss:  0.06217| Train acc:  0.98065| Test loss:  0.05944| Test acc:  0.98063
done


In [11]:
total_mean = mean(array(times["load"]) + array(times["device"]) + array(times["eval_acc"]) + array(times["bp"]))
print(f"each for loop takes an average of {total_mean}s")
mean(times["load"])/total_mean, mean(times["device"])/total_mean, mean(times["eval_acc"])/total_mean, mean(times["bp"])/total_mean

each for loop takes an average of 0.004945059585571289s


(0.5406173193500328,
 0.01131517888451779,
 0.20829144223930085,
 0.23977605952614853)

6.6 ms each loop. 42% of the time was spent **loading data**

In [12]:
std(times["load"]), std(times["device"]), std(times["eval_acc"]), std(times["bp"])

(0.0005959969163981126,
 2.9378703515497135e-05,
 0.00808939392496398,
 0.004175204962222429)

## Load all MNIST at once

In [13]:
alltrainloader = torch.utils.data.DataLoader(trainval_dataset, batch_size=len(trainval_dataset), shuffle=True)
alltest_dataloader = DataLoader(dataset=test_dataset, batch_size=len(test_dataset), shuffle=True)

print("Loading all training dataset...", end="")
alltrain_iter = iter(alltrainloader)
alltrain_images, alltrain_labels = next(alltrain_iter)
print("Loaded")
print("Loading all test dataset...", end="")
alltest_iter = iter(alltest_dataloader)
alltest_images, alltest_labels = next(alltest_iter)
print("Loaded")

Loading all training dataset...Loaded
Loadedg all test dataset...


In [14]:
alltrain_images.shape, alltest_images.shape, alltrain_labels.shape, alltest_labels.shape

(torch.Size([60000, 1, 28, 28]),
 torch.Size([10000, 1, 28, 28]),
 torch.Size([60000]),
 torch.Size([10000]))

In [15]:
times = dict(load=[], device=[], eval_acc=[], bp=[])
times

{'load': [], 'device': [], 'eval_acc': [], 'bp': []}

In [16]:
# device-agnostic setup
device = 'cuda' if torch.cuda.is_available() else 'cpu'
accuracy = accuracy.to(device)
model_lenet5v1 = model_lenet5v1.to(device)

EPOCHS = 2

# for epoch in tqdm(range(EPOCHS)):
for epoch in range(EPOCHS):
    print(f"epoch - {epoch}")
    # Training loop
    train_loss, train_acc = 0.0, 0.0
    start = time.time()
    pos = 0
    next_pos=pos+BATCH_SIZE if pos+BATCH_SIZE<len(alltrain_images) else len(alltrain_images)
    while pos<len(alltrain_images):
    # for X, y in zip(alltrain_images[pos:next_pos], alltrain_labels[pos:next_pos]):
        X, y = alltrain_images[pos:next_pos], alltrain_labels[pos:next_pos]
    # for X, y in trainval_dataloader:
        # print(X.shape, y.shape)
        # print(y)
        t_load = time.time()
        X, y = X.to(device), y.to(device)
        t_device = time.time()
        
        model_lenet5v1.train()
        
        y_pred = model_lenet5v1(X)
        
        loss = loss_fn(y_pred, y)
        train_loss += loss.item()
        
        acc = accuracy(y_pred, y)
        train_acc += acc
        t_eval_acc = time.time()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        t_bp = time.time()

        times["load"].append(t_load-start)
        times["device"].append(t_device-t_load)
        times["eval_acc"].append(t_eval_acc-t_device)
        times["bp"].append(t_bp-t_eval_acc)
        # print(f"t_load={t_load-start}\nt_device={t_device-t_load}\nt_eval_acc={t_eval_acc-t_device}\nt_bp={t_bp-t_eval_acc}")
        pos+=BATCH_SIZE
        next_pos= (pos+BATCH_SIZE) if (pos+BATCH_SIZE)<=len(alltrain_images) else len(alltrain_images)
        start = time.time()

        
    train_loss /= len(trainval_dataloader)
    train_acc /= len(trainval_dataloader)
        
    # Validation loop
    val_loss, val_acc = 0.0, 0.0
    model_lenet5v1.eval()
    with torch.inference_mode():
        # for X, y in test_dataloader:
        # for X, y in zip(alltest_images, alltest_labels):
        pos = 0
        next_pos=pos+BATCH_SIZE if pos+BATCH_SIZE<len(alltest_images) else len(alltest_images)
        while pos<len(alltest_images):
            X, y = alltest_images[pos:next_pos], alltest_labels[pos:next_pos]
            X, y = X.to(device), y.to(device)
            
            y_pred = model_lenet5v1(X)
            
            loss = loss_fn(y_pred, y)
            val_loss += loss.item()
            
            acc = accuracy(y_pred, y)
            val_acc += acc
            pos+=BATCH_SIZE
            next_pos= (pos+BATCH_SIZE) if (pos+BATCH_SIZE)<=len(alltest_images) else len(alltest_images)
            
        val_loss /= len(test_dataloader)
        val_acc /= len(test_dataloader)
        
    print(f"Epoch: {epoch}| Train loss: {train_loss: .5f}| Train acc: {train_acc: .5f}| Test loss: {val_loss: .5f}| Test acc: {val_acc: .5f}")
print("done")

epoch - 0
Epoch: 0| Train loss:  0.04648| Train acc:  0.98573| Test loss:  0.04074| Test acc:  0.98712
epoch - 1
Epoch: 1| Train loss:  0.03470| Train acc:  0.98875| Test loss:  0.03825| Test acc:  0.98732
done


In [17]:
total_mean = mean(array(times["load"]) + array(times["device"]) + array(times["eval_acc"]) + array(times["bp"]))
print(f"each for loop takes an average of {total_mean}s")
mean(times["load"])/total_mean, mean(times["device"])/total_mean, mean(times["eval_acc"])/total_mean, mean(times["bp"])/total_mean

each for loop takes an average of 0.002398253059387207s


(0.0052567704593317,
 0.033184629388960746,
 0.35634179124296994,
 0.6052168089087376)

Each loop took 2.8ms. **More than 2x faster.**

## Profiling from python

In [18]:
import cProfile

In [19]:
def fn():
    cumsum([j*sum([i*13 for i in range(10000)]) 
         for j in range(300)])

In [20]:
%prun -s cumulative fn()
# %prun fn()

         2712 function calls in 0.219 seconds

   Ordered by: cumulative time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    0.229    0.229 {built-in method builtins.exec}
        1    0.000    0.000    0.229    0.229 <string>:1(<module>)
        1    0.000    0.000    0.229    0.229 2310747687.py:1(fn)
      300    0.000    0.000    0.119    0.000 fromnumeric.py:2177(sum)
      300    0.001    0.000    0.118    0.000 fromnumeric.py:71(_wrapreduction)
      300    0.117    0.000    0.117    0.000 {method 'reduce' of 'numpy.ufunc' objects}
      300    0.100    0.000    0.100    0.000 2310747687.py:2(<listcomp>)
      300    0.000    0.000    0.000    0.000 fromnumeric.py:2172(_sum_dispatcher)
      302    0.000    0.000    0.000    0.000 {built-in method builtins.getattr}
      300    0.000    0.000    0.000    0.000 fromnumeric.py:72(<dictcomp>)
        1    0.000    0.000    0.000    0.000 fromnumeric.py:2512(cumsum)
        1 

Try `snakeviz` for visualization in a pie chart
https://stackoverflow.com/a/37431235/1273751